# Pipeline de ETL: Limpieza, Modelado y Carga de Datos de Ventas

## 1. Introducción y Contexto del Proyecto
Este notebook documenta el proceso de **Extracción, Transformación y Carga (ETL)** realizado para un proyecto de analítica de negocio. El objetivo principal es procesar un conjunto de datos transaccionales de ventas que presenta inconsistencias, nulos y duplicados, transformarlo en un modelo limpio y optimizado, y posteriormente cargarlo en una base de datos relacional para su visualización final en un dashboard.

### Objetivos Técnicos:
* **Calidad de Datos:** Identificar y corregir valores faltantes, registros duplicados y estandarizar formatos de texto, fechas y horas.
* **Ingeniería de Características:** Generar nuevas métricas clave de negocio (como la ganancia neta) directamente en el backend para optimizar el rendimiento del reporte.
* **Carga de Datos:** Diseñar un pipeline automatizado para persistir la información estructurada en una base de datos SQLite.

In [20]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

# Load the data
try:
    data = pd.read_excel('data/raw/Ventas_Crudas.xlsx')
except Exception as e:
    print(f"Error loading data from {'Ventas_Crudas.xlsx'}: {e}")

## 2. Análisis Exploratorio de Datos (EDA) e Identificación de Inconsistencias
Antes de aplicar cualquier transformación, realizamos una inspección inicial del set de datos crudo (`Ventas_Crudas.xlsx`) para evaluar la integridad de la información, el tipo de variables y la presencia de anomalías que puedan sesgar los reportes en Power BI.

In [21]:
data.info()
print(f'\nNull data: \n{data.isnull().sum()}')
print(f'\nDuplicated data: {data.duplicated().sum()}')
data.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   ID_Ticket      201 non-null    object        
 1   Fecha          200 non-null    datetime64[ns]
 2   Hora           200 non-null    object        
 3   Categoría      196 non-null    object        
 4   Producto       194 non-null    object        
 5   Cantidad       194 non-null    float64       
 6   Precio_Venta   195 non-null    float64       
 7   Costo_Insumos  195 non-null    float64       
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 12.7+ KB

Null data: 
ID_Ticket        0
Fecha            1
Hora             1
Categoría        5
Producto         7
Cantidad         7
Precio_Venta     6
Costo_Insumos    6
dtype: int64

Duplicated data: 0


,ID_Ticket,Fecha,Hora,Categoría,Producto,Cantidad,Precio_Venta,Costo_Insumos
0,1,2026-07-22,10:58,Comida,Pizza,1.0,125.0,75.0
1,2,2026-08-01,16:28,Comida,Hamburguesa,1.0,100.0,50.0
2,3,2026-08-20,10:30,Postre,Helado,3.0,225.0,105.0
3,4,2026-07-15,15:3,Postre,Helado,3.0,225.0,105.0
4,5,2026-08-07,13:8,Bebida,Agua,1.0,30.0,15.0


## 3. Fase de Limpieza y Transformación de Datos (ETL)

Procedemos a aplicar las reglas de transformación definidas para asegurar la consistencia del modelo de datos:

1. **Remoción de registros inválidos:** Eliminación de filas completamente vacías (`dropna`) y transacciones repetidas (`drop_duplicates`).
2. **Estandarización de nomenclatura:** Conversión de los nombres de todas las columnas a formato *snake_case* y minúsculas para facilitar las consultas SQL posteriores.
3. **Normalización de Texto:** Limpieza de espacios en blanco al inicio y final (`.str.strip()`), eliminación de acentos y conversión a minúsculas en columnas categóricas como `categoría` y `producto`. Esto evita que en el dashboard se dupliquen categorías por errores de dedo (ej. "Bebidas" vs "bebidas ").
4. **Formateo Temporal:** Conversión estricta de las columnas `fecha` y `hora` a tipos de datos temporales nativos de Python.

In [22]:
data = data.dropna() # Clean Null Data
data = data.drop_duplicates() # Clean Duplicated Data
data.columns = data.columns.str.lower().str.strip().str.replace(' ', '_') # Clean all the titles
data.rename(columns={'categoría': 'categoria'}, inplace=True) 

# Clean specific columns
columns = ['categoria', 'producto']
for col in columns:
    data[col] = data[col].str.lower().str.strip().str.replace(' ', '_')

In [23]:
data['fecha'] = pd.to_datetime(data['fecha'], errors='coerce').dt.date  # Clean fecha column
data['hora'] = pd.to_datetime(data['hora'], format='%H:%M', errors='coerce').dt.strftime('%H:%M') # Convert hora to time format

## 4. Ingeniería de Características (Feature Engineering)
Para optimizar el rendimiento del modelo de datos en el dashboard y evitar calcular columnas complejas mediante DAX en Power BI (lo cual impactaría la fluidez del reporte), calculamos la métrica de negocio **Ganancia Neta** directamente en esta etapa utilizando la lógica:

$$\text{Ganancia Neta} = \text{Precio de Venta} - \text{Costo de Insumos}$$

In [24]:
data['ganancia_neta'] = data['precio_venta'] - data['costo_insumos']  # Calculate net profit
data.head()

,id_ticket,fecha,hora,categoria,producto,cantidad,precio_venta,costo_insumos,ganancia_neta
0,1,2026-07-22,10:58,comida,pizza,1.0,125.0,75.0,50.0
1,2,2026-08-01,16:28,comida,hamburguesa,1.0,100.0,50.0,50.0
2,3,2026-08-20,10:30,postre,helado,3.0,225.0,105.0,120.0
3,4,2026-07-15,15:03,postre,helado,3.0,225.0,105.0,120.0
4,5,2026-08-07,13:08,bebida,agua,1.0,30.0,15.0,15.0


## 5. Almacenamiento y Carga de Datos (Load)
Una vez que los datos han sido validados y estructurados correctamente, el pipeline utiliza `SQLAlchemy` para conectarse a un entorno local de **SQLite** (`ventas.db`). 

La información procesada se almacena en la tabla relacional `historial_ventas` aplicando una estrategia de reemplazo (`if_exists='replace'`) para garantizar que el backend del dashboard consuma siempre la versión de datos más actualizada y depurada.

In [27]:
db_url = 'sqlite:///database/ventas.db'
table_name = 'historial_ventas'

try:
    engine = create_engine(db_url)
    data.to_sql(table_name, con=engine, if_exists='replace', index=False)
    data.to_excel('data/processed/Ventas_Limpias.xlsx', index=False)
    print(f"Data saved to database table '{table_name}' successfully.")
    print(f"Data saved in a new xlsx 'Ventas_Limpias' successfully.")
except Exception as e:
    print(f"Error saving data: {e}")

Data saved to database table 'historial_ventas' successfully.
Data saved in a new xlsx 'Ventas_Limpias' successfully.
